Data treatment for experiment.

In [25]:
import numpy as np
import pandas as pd

# custom libs
import sys
sys.path.append("..")
from src.data_wrangling import add_return_cols, ts_to_df
from src.utils import mark_repeated_columns

In [26]:
# Data
data_path = "../data/processed/"
returns_path = ''.join([data_path, 'btc.xlsx'])
df = pd.read_excel(returns_path).loc[:, ["datetime", "value"]]
df.head()

,datetime,value
0,2024-12-02 00:10:00,587204.12
1,2024-12-02 00:15:00,587405.84
2,2024-12-02 00:20:00,587333.97
3,2024-12-02 00:25:00,586735.29
4,2024-12-02 00:30:00,586787.35


In [27]:
print("Min hour:", df["datetime"].dt.time.min())
print("Max hour:", df["datetime"].dt.time.max())

Min hour: 00:00:00
Max hour: 23:55:00


In [28]:
def validate_intraday_data(df):
    df = df.copy()
    
    print("="*60)
    print("🔍 INTRADAY DATA VALIDATION REPORT")
    print("="*60)

    # --- Basic setup ---
    df['datetime'] = pd.to_datetime(df['datetime'])
    df = df.sort_values('datetime').reset_index(drop=True)
    df['date'] = df['datetime'].dt.date

    # --- 1. Timestamp integrity ---
    print("\n📌 1. TIMESTAMP INTEGRITY")
    dup_ts = df['datetime'].duplicated().sum()
    print(f"Duplicate timestamps: {dup_ts}")

    full_range = pd.date_range(df['datetime'].min(), df['datetime'].max(), freq='5min')
    missing_ts = full_range.difference(df['datetime'])
    print(f"Missing timestamps: {len(missing_ts)}")

    time_diff = df['datetime'].diff().dropna()
    irregular = (time_diff != pd.Timedelta(minutes=5)).sum()
    print(f"Irregular intervals: {irregular}")

    # --- 2. Daily structure ---
    print("\n📅 2. DAILY STRUCTURE")
    counts = df.groupby('date').size()
    print("Observations per day (summary):")
    print(counts.describe())

    bad_counts = counts[counts != 288]
    print(f"Days with != 288 obs: {len(bad_counts)}")

    daily_bounds = df.groupby('date')['datetime'].agg(['min', 'max'])
    bad_bounds = daily_bounds[
        (daily_bounds['min'].dt.time != pd.to_datetime("00:00").time()) |
        (daily_bounds['max'].dt.time != pd.to_datetime("23:55").time())
    ]
    print(f"Days with irregular start/end: {len(bad_bounds)}")

    # --- 3. Duplicates & repeated values ---
    print("\n🔁 3. DUPLICATES & REPEATED VALUES")
    dup_rows = df.duplicated().sum()
    print(f"Duplicate rows: {dup_rows}")

    same_price = (df['value'].diff() == 0).sum()
    print(f"Consecutive identical prices: {same_price}")

    # --- 4. Missing / invalid values ---
    print("\n📉 4. MISSING / INVALID VALUES")
    na_counts = df.isna().sum()
    print("NaNs per column:")
    print(na_counts)

    invalid_prices = (df['value'] <= 0).sum()
    print(f"Non-positive prices: {invalid_prices}")

    # --- 5. Returns & outliers ---
    print("\n⚠️ 5. RETURNS & OUTLIERS")
    df['log_return'] = np.log(df['value']).diff()
    print(df['log_return'].describe())

    threshold = 5 * df['log_return'].std()
    extremes = (np.abs(df['log_return']) > threshold).sum()
    print(f"Extreme returns (>5σ): {extremes}")

    # --- 6. Structural checks ---
    print("\n🧠 6. STRUCTURAL CHECKS")
    print(f"Sorted: {df['datetime'].is_monotonic_increasing}")
    print(f"Timezone: {df['datetime'].dt.tz}")
    print("Dtypes:")
    print(df.dtypes)

    # --- 7. Intraday grid consistency ---
    print("\n🧩 7. INTRADAY GRID CONSISTENCY")
    intraday_times = df['datetime'].dt.time
    ref_day = df['date'].iloc[0]
    reference = set(intraday_times[df['date'] == ref_day])

    bad_days = [
        d for d in df['date'].unique()
        if set(intraday_times[df['date'] == d]) != reference
    ]

    print(f"Days with inconsistent grids: {len(bad_days)}")

    # --- Final summary ---
    print("\n" + "="*60)
    print("📊 FINAL DIAGNOSTIC SUMMARY")
    print("="*60)

    issues = {
        "duplicate_timestamps": dup_ts,
        "missing_timestamps": len(missing_ts),
        "irregular_intervals": irregular,
        "bad_day_counts": len(bad_counts),
        "bad_day_bounds": len(bad_bounds),
        "duplicate_rows": dup_rows,
        "invalid_prices": invalid_prices,
        "extreme_returns": extremes,
        "inconsistent_intraday_days": len(bad_days)
    }

    for k, v in issues.items():
        status = "✅ OK" if v == 0 else "❌ ISSUE"
        print(f"{k:<35} {v:<10} {status}")

    print("="*60)

    return {
        "missing_timestamps": missing_ts,
        "bad_days_counts": bad_counts,
        "bad_days_bounds": bad_bounds,
        "extreme_returns": df[np.abs(df['log_return']) > threshold],
        "inconsistent_days": bad_days
    }

In [29]:
results = validate_intraday_data(df)

🔍 INTRADAY DATA VALIDATION REPORT

📌 1. TIMESTAMP INTEGRITY
Duplicate timestamps: 0
Missing timestamps: 34921
Irregular intervals: 196

📅 2. DAILY STRUCTURE
Observations per day (summary):
count    300.000000
mean     233.030000
std       91.378311
min       12.000000
25%      251.750000
50%      286.000000
75%      288.000000
max      288.000000
dtype: float64
Days with != 288 obs: 184
Days with irregular start/end: 144

🔁 3. DUPLICATES & REPEATED VALUES
Duplicate rows: 0
Consecutive identical prices: 1589

📉 4. MISSING / INVALID VALUES
NaNs per column:
datetime    0
value       0
date        0
dtype: int64
Non-positive prices: 0

⚠️ 5. RETURNS & OUTLIERS
count    69908.000000
mean        -0.000003
std          0.001826
min         -0.065602
25%         -0.000745
50%          0.000000
75%          0.000743
max          0.105846
Name: log_return, dtype: float64
Extreme returns (>5σ): 188

🧠 6. STRUCTURAL CHECKS
Sorted: True
Timezone: None
Dtypes:
datetime      datetime64[us]
value     

In [31]:
df.set_index("datetime")["2024-12-02 07:00:00":"2024-12-02 08:30:00"]

,value
datetime,
2024-12-02 07:00:00,574225.39
2024-12-02 07:05:00,573754.11
2024-12-02 07:10:00,573748.45
2024-12-02 07:15:00,572572.56
2024-12-02 07:20:00,572642.94
2024-12-02 07:25:00,572771.66
2024-12-02 07:30:00,572947.70
2024-12-02 08:10:00,572000.00
2024-12-02 08:15:00,573594.00


In [64]:
df2 = df.copy().reset_index()

df2['datetime'] = pd.to_datetime(df['datetime'])
df2 = df.sort_values('datetime')

df2['weekday'] = df['datetime'].dt.weekday
df2['time'] = df['datetime'].dt.time

df2 = df2[
    (df2['weekday'] < 5) &  # Monday-Friday
    (df2['time'] >= pd.to_datetime("10:00").time()) &
    (df2['time'] <= pd.to_datetime("17:00").time())
]

df2 = df2.set_index('datetime')

full_grid = pd.date_range(
    df2.index.min().floor('D') + pd.Timedelta(hours=10),
    df2.index.max().floor('D') + pd.Timedelta(hours=17),
    freq='5min'
)

# Keep only weekdays
full_grid = full_grid[full_grid.weekday < 5]

df2 = df2.reindex(full_grid)

df2['date'] = df2.index.date

counts = df2.groupby('date')['value'].count()
good_days = counts[counts == 85].index

df2 = df2[df2['date'].isin(good_days)].reset_index()
df2.rename(columns={"index": "datetime"}, inplace=True)
# df2 = df2[["index", "value"]]
# df2.columns = [["datetime", "value"]]
# df2

In [65]:
def validate_intraday_10_17(df):
    import pandas as pd
    import numpy as np

    df = df.copy()
    df['datetime'] = pd.to_datetime(df['datetime'])
    df = df.sort_values('datetime')

    # --- Filter context: weekdays + 10:00–17:00 (inclusive) ---
    df['weekday'] = df['datetime'].dt.weekday
    df['time'] = df['datetime'].dt.time
    df['date'] = df['datetime'].dt.date

    df = df[
        (df['weekday'] < 5) &
        (df['time'] >= pd.to_datetime("10:00").time()) &
        (df['time'] <= pd.to_datetime("17:00").time())
    ]

    EXPECTED = 85

    print("="*65)
    print("🔍 INTRADAY VALIDATION (WEEKDAYS 10:00–17:00)")
    print(f"Expected observations per day: {EXPECTED}")
    print("="*65)

    # --- 1. Counts per day ---
    counts = df.groupby('date').size()

    print("\n📊 1. OBSERVATIONS PER DAY")
    print(counts.describe())

    bad_counts = counts[counts != EXPECTED]
    print(f"\nDays with != {EXPECTED} observations: {len(bad_counts)}")

    # --- 2. Start / End times ---
    print("\n⏱️ 2. DAILY BOUNDS CHECK")
    bounds = df.groupby('date')['datetime'].agg(['min', 'max'])

    bad_bounds = bounds[
        (bounds['min'].dt.time != pd.to_datetime("10:00").time()) |
        (bounds['max'].dt.time != pd.to_datetime("17:00").time())
    ]

    print(f"Days with incorrect start/end: {len(bad_bounds)}")

    # --- 3. Grid consistency ---
    print("\n🧩 3. GRID CONSISTENCY (CRITICAL)")

    intraday_times = df['datetime'].dt.time
    ref_day = counts.index[0]
    reference_grid = set(intraday_times[df['date'] == ref_day])

    inconsistent_days = [
        d for d in counts.index
        if set(intraday_times[df['date'] == d]) != reference_grid
    ]

    print(f"Inconsistent intraday grids: {len(inconsistent_days)}")

    # --- 4. Missing timestamps vs ideal grid ---
    print("\n🕳️ 4. MISSING TIMESTAMPS")

    # Build ideal grid
    unique_days = sorted(counts.index)
    full_grid = []

    for d in unique_days:
        day_start = pd.Timestamp(d) + pd.Timedelta(hours=10)
        day_grid = pd.date_range(day_start, periods=EXPECTED, freq='5min')
        full_grid.append(day_grid)

    full_grid = full_grid[0].append(full_grid[1:]) if len(full_grid) > 1 else full_grid[0]

    df_indexed = df.set_index('datetime')
    missing_ts = full_grid.difference(df_indexed.index)

    print(f"Missing timestamps (within window): {len(missing_ts)}")

    # --- 5. Returns & extremes ---
    print("\n⚠️ 5. RETURNS DIAGNOSTICS")

    df['log_return'] = np.log(df['value']).diff()
    desc = df['log_return'].describe()
    print(desc)

    threshold = 5 * df['log_return'].std()
    extreme_returns = df[np.abs(df['log_return']) > threshold]

    print(f"Extreme returns (>5σ): {len(extreme_returns)}")

    # --- 6. Flat prices ---
    print("\n🔁 6. PRICE STAGNATION")

    same_price = (df['value'].diff() == 0).sum()
    print(f"Consecutive identical prices: {same_price}")

    # --- FINAL SUMMARY ---
    print("\n" + "="*65)
    print("📊 FINAL SUMMARY")
    print("="*65)

    issues = {
        "bad_day_counts": len(bad_counts),
        "bad_day_bounds": len(bad_bounds),
        "inconsistent_grids": len(inconsistent_days),
        "missing_timestamps": len(missing_ts),
        "extreme_returns": len(extreme_returns),
    }

    for k, v in issues.items():
        status = "✅ OK" if v == 0 else "❌ ISSUE"
        print(f"{k:<30} {v:<10} {status}")

    print("="*65)

    # --- Return detailed objects for inspection ---
    return {
        "filtered_df": df,
        "bad_counts": bad_counts,
        "bad_bounds": bad_bounds,
        "inconsistent_days": inconsistent_days,
        "missing_timestamps": missing_ts,
        "extreme_returns": extreme_returns
    }

In [66]:
results = validate_intraday_10_17(df2)

🔍 INTRADAY VALIDATION (WEEKDAYS 10:00–17:00)
Expected observations per day: 85

📊 1. OBSERVATIONS PER DAY
count    239.0
mean      85.0
std        0.0
min       85.0
25%       85.0
50%       85.0
75%       85.0
max       85.0
dtype: float64

Days with != 85 observations: 0

⏱️ 2. DAILY BOUNDS CHECK
Days with incorrect start/end: 0

🧩 3. GRID CONSISTENCY (CRITICAL)
Inconsistent intraday grids: 0

🕳️ 4. MISSING TIMESTAMPS
Missing timestamps (within window): 0

⚠️ 5. RETURNS DIAGNOSTICS
count    20314.000000
mean        -0.000008
std          0.003049
min         -0.075480
25%         -0.001059
50%          0.000000
75%          0.001013
max          0.092556
Name: log_return, dtype: float64
Extreme returns (>5σ): 106

🔁 6. PRICE STAGNATION
Consecutive identical prices: 149

📊 FINAL SUMMARY
bad_day_counts                 0          ✅ OK
bad_day_bounds                 0          ✅ OK
inconsistent_grids             0          ✅ OK
missing_timestamps             0          ✅ OK
extreme_retur

In [52]:
df2["2024-12-02 10:00:00":"2024-12-02 17:00:00"].to_clipboard()

In [34]:
results2 = validate_intraday_data(df2)

🔍 INTRADAY DATA VALIDATION REPORT

📌 1. TIMESTAMP INTEGRITY
Duplicate timestamps: 0
Missing timestamps: 83039
Irregular intervals: 255

📅 2. DAILY STRUCTURE
Observations per day (summary):
count    252.000000
mean      83.388889
std        8.428724
min        9.000000
25%       85.000000
50%       85.000000
75%       85.000000
max       85.000000
dtype: float64
Days with != 288 obs: 252
Days with irregular start/end: 252

🔁 3. DUPLICATES & REPEATED VALUES
Duplicate rows: 0
Consecutive identical prices: 156

📉 4. MISSING / INVALID VALUES
NaNs per column:
datetime    0
value       0
weekday     0
time        0
date        0
dtype: int64
Non-positive prices: 0

⚠️ 5. RETURNS & OUTLIERS
count    21013.000000
mean        -0.000008
std          0.003046
min         -0.075480
25%         -0.001060
50%          0.000000
75%          0.001022
max          0.092556
Name: log_return, dtype: float64
Extreme returns (>5σ): 111

🧠 6. STRUCTURAL CHECKS
Sorted: True
Timezone: None
Dtypes:
datetime    

In [115]:
df3 = df2[["datetime", "value"]].dropna()

In [116]:
# creates return columns
df_returns = add_return_cols(df=df3, y="value", dropna=True)

# turns long to wide format (dates become columns)
df_returns_wide = ts_to_df(
                        df_returns, 
                        y='R_t', 
                        datetime_col='datetime'
                        ) \
                        .iloc[1:,:] # removes overnight return
                        
df_returns_wide.head()

date,2024-12-02,2024-12-03,2024-12-04,2024-12-12,2024-12-13,2024-12-16,2024-12-17,2024-12-18,2024-12-19,2024-12-23,...,2025-11-17,2025-11-18,2025-11-19,2025-11-20,2025-11-21,2025-11-24,2025-11-25,2025-11-26,2025-11-27,2025-11-28
time,,,,,,,,,,,,,,,,,,,,,
10:05:00,0.003534,-0.001926,0.000362,0.002040,-0.001221,0.000579,0.000179,0.000744,0.001014,-0.001643,...,-0.001805,-0.004055,0.000190,0.000406,0.008330,0.003560,-0.000166,-0.001121,-0.001003,-0.001000
10:10:00,0.001567,-0.002448,0.000237,-0.001097,-0.001201,0.002135,0.000575,0.002471,0.002589,-0.003237,...,-0.001899,0.000588,-0.001064,0.003135,-0.000495,0.002794,0.002346,0.000706,-0.000560,0.002336
10:15:00,0.003157,-0.001255,-0.001031,0.002720,-0.000656,-0.001271,0.000716,-0.000267,-0.000456,0.001479,...,0.000769,-0.000899,0.002051,-0.002503,0.004562,-0.001615,-0.000344,0.000195,-0.002083,0.001334
10:20:00,-0.000784,0.001392,-0.001087,0.001268,-0.000894,0.000690,0.001277,-0.000398,-0.002377,-0.001397,...,-0.000294,-0.001681,-0.000692,0.001162,-0.005734,0.000450,-0.000461,0.000613,-0.000620,0.000043
10:25:00,0.001506,-0.000744,-0.001100,-0.000192,-0.000392,-0.001605,0.003030,-0.000028,-0.002171,-0.000091,...,-0.003111,-0.001168,0.000517,0.000723,-0.002590,0.001865,-0.000357,0.000394,0.002006,0.000502


In [123]:
df.set_index("datetime")["2025-11-25 10:00:00":"2025-11-25 17:05:00"]

,value
datetime,
2025-11-25 10:00:00,469366.0
2025-11-25 10:05:00,469288.0
2025-11-25 10:10:00,470389.0
2025-11-25 10:15:00,470227.0
2025-11-25 10:20:00,470010.0
...,...
2025-11-25 16:45:00,470833.0
2025-11-25 16:50:00,470833.0
2025-11-25 16:55:00,470833.0


In [117]:
df_returns_wide.tail()

date,2024-12-02,2024-12-03,2024-12-04,2024-12-12,2024-12-13,2024-12-16,2024-12-17,2024-12-18,2024-12-19,2024-12-23,...,2025-11-17,2025-11-18,2025-11-19,2025-11-20,2025-11-21,2025-11-24,2025-11-25,2025-11-26,2025-11-27,2025-11-28
time,,,,,,,,,,,,,,,,,,,,,
16:40:00,0.000547,-0.001918,0.001310,-0.002297,0.000202,0.000326,-0.000608,0.000053,-0.004508,0.000696,...,-0.000335,-0.003992,-0.000002,0.001800,-0.004454,-0.005041,0.0,0.000044,0.000069,0.000909
16:45:00,0.000259,0.001343,-0.000431,0.001184,0.000501,-0.000111,0.001021,-0.003260,-0.002694,0.000252,...,-0.001308,-0.001159,0.000059,-0.001883,0.000204,-0.000478,0.0,-0.001269,-0.001135,-0.000010
16:50:00,-0.002402,-0.000275,0.002676,0.000122,0.002468,0.001774,-0.001526,0.001133,-0.001077,-0.001544,...,-0.000499,-0.000270,0.002723,0.002033,-0.002397,-0.002163,0.0,0.002235,-0.000543,0.000111
16:55:00,0.002457,0.001663,0.002865,-0.004320,-0.001408,0.000901,0.000670,-0.000978,0.004574,0.000865,...,-0.000010,0.001238,-0.001688,0.003114,0.000187,0.002964,0.0,-0.000504,0.000421,0.000392
17:00:00,0.003137,0.000461,0.000074,0.000518,0.001122,-0.002341,0.000130,-0.003498,-0.002960,-0.000082,...,-0.002449,0.000565,-0.001482,-0.002842,0.000206,0.001449,0.0,-0.001550,-0.000358,-0.001376


In [137]:
# Removes inconsistent days
columns_to_drop, columns_to_keep= mark_repeated_columns(df_returns_wide)
print(f"Dropping {len(columns_to_drop)} columns due to possible incomplete data:\n", columns_to_drop)
df_returns_wide_cleaned = df_returns_wide.loc[:, ~columns_to_keep]

Dropping 3 columns due to possible incomplete data:
 [datetime.date(2025, 5, 12), datetime.date(2025, 5, 16), datetime.date(2025, 11, 25)]


In [142]:
df_returns_wide_cleaned.head()

date,2024-12-02,2024-12-03,2024-12-04,2024-12-12,2024-12-13,2024-12-16,2024-12-17,2024-12-18,2024-12-19,2024-12-23,...,2025-11-14,2025-11-17,2025-11-18,2025-11-19,2025-11-20,2025-11-21,2025-11-24,2025-11-26,2025-11-27,2025-11-28
time,,,,,,,,,,,,,,,,,,,,,
10:05:00,0.003534,-0.001926,0.000362,0.002040,-0.001221,0.000579,0.000179,0.000744,0.001014,-0.001643,...,-0.002870,-0.001805,-0.004055,0.000190,0.000406,0.008330,0.003560,-0.001121,-0.001003,-0.001000
10:10:00,0.001567,-0.002448,0.000237,-0.001097,-0.001201,0.002135,0.000575,0.002471,0.002589,-0.003237,...,-0.001696,-0.001899,0.000588,-0.001064,0.003135,-0.000495,0.002794,0.000706,-0.000560,0.002336
10:15:00,0.003157,-0.001255,-0.001031,0.002720,-0.000656,-0.001271,0.000716,-0.000267,-0.000456,0.001479,...,0.000036,0.000769,-0.000899,0.002051,-0.002503,0.004562,-0.001615,0.000195,-0.002083,0.001334
10:20:00,-0.000784,0.001392,-0.001087,0.001268,-0.000894,0.000690,0.001277,-0.000398,-0.002377,-0.001397,...,-0.001757,-0.000294,-0.001681,-0.000692,0.001162,-0.005734,0.000450,0.000613,-0.000620,0.000043
10:25:00,0.001506,-0.000744,-0.001100,-0.000192,-0.000392,-0.001605,0.003030,-0.000028,-0.002171,-0.000091,...,0.002401,-0.003111,-0.001168,0.000517,0.000723,-0.002590,0.001865,0.000394,0.002006,0.000502


In [146]:
df_returns_wide_cleaned.to_excel("../data/processed/btc_treated.xlsx")

In [145]:
df_returns = df_returns[~(df_returns["datetime"].dt.normalize().isin(pd.to_datetime(columns_to_drop)))].loc[:,["datetime","value","R_t","r_t"]]
df_returns.to_excel("../data/processed/btc_treated_long.xlsx", index=False)